In [31]:
import pandas as pd
import numpy as np
import requests

In [32]:
url = "https://api.pbpstats.com/get-totals/nba"
params = {
    "Season": "2024-25",
    "SeasonType": "Regular Season",
    "Type": "Player"
}
response = requests.get(url, params=params)
response_json = response.json()
player_stats = response_json["multi_row_table_data"]

In [33]:
player_stats_df = pd.DataFrame(player_stats)
player_stats_df

,EntityId,TeamId,Name,ShortName,RowId,TeamAbbreviation,SecondsPlayed,GamesPlayed,Minutes,PlusMinus,...,Corner3PctBlocked,Arc3PctBlocked,OffFTReboundPct,Clear Path Fouls,3pt And 1 Free Throw Trips,BlockedLongMidRange,HeaveMakes,3SecondViolations,Period3Fouls5Minutes,Period1Fouls3Minutes
0,1628969,1610612752,Mikal Bridges,Mikal Bridges,1628969,NYK,182161.0,82,3036,334.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1628404,1610612752,Josh Hart,Josh Hart,1628404,NYK,173809.0,77,2897,203.0,...,0.015152,0.016129,0.016949,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1630162,1610612750,Anthony Edwards,Anthony Edwards,1630162,MIN,172266.0,79,2871,291.0,...,0.030000,0.002813,0.014706,1.0,2.0,3.0,NaN,NaN,NaN,NaN
3,1626164,1610612756,Devin Booker,Devin Booker,1626164,PHX,167699.0,75,2795,-177.0,...,NaN,0.004202,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN
4,201935,1610612746,James Harden,James Harden,201935,LAC,167350.0,79,2789,344.0,...,0.033333,0.004702,0.014493,NaN,11.0,1.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,1631303,1610612757,Justin Minaya,Justin Minaya,1631303,POR,6040.0,19,101,-12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
496,1642352,1610612748,Keshad Johnson,Keshad Johnson,1642352,MIA,5887.0,16,98,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497,1642530,1610612763,Yuki Kawamura,Yuki Kawamura,1642530,MEM,5559.0,22,93,27.0,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
498,1631247,1610612739,Luke Travers,Luke Travers,1631247,CLE,5271.0,12,88,-1.0,...,0.200000,NaN,0.166667,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
for col in player_stats_df.columns:
    print(col)

EntityId
TeamId
Name
ShortName
RowId
TeamAbbreviation
SecondsPlayed
GamesPlayed
Minutes
PlusMinus
OffPoss
DefPoss
PenaltyOffPoss
PenaltyDefPoss
SecondChanceOffPoss
TotalPoss
AtRimFGM
AtRimFGA
SecondChanceAtRimFGM
SecondChanceAtRimFGA
PenaltyAtRimFGM
PenaltyAtRimFGA
ShortMidRangeFGM
ShortMidRangeFGA
LongMidRangeFGM
LongMidRangeFGA
Corner3FGM
Corner3FGA
SecondChanceCorner3FGM
SecondChanceCorner3FGA
PenaltyCorner3FGM
PenaltyCorner3FGA
Arc3FGM
Arc3FGA
SecondChanceArc3FGM
SecondChanceArc3FGA
PenaltyArc3FGM
PenaltyArc3FGA
FG2M
FG2A
FG3M
FG3A
FtPoints
Points
OpponentPoints
SecondChanceFG2M
SecondChanceFG2A
SecondChanceFG3M
SecondChanceFG3A
SecondChanceFtPoints
SecondChancePoints
PenaltyFG2M
PenaltyFG2A
PenaltyFG3M
PenaltyFG3A
PenaltyFtPoints
PenaltyPoints
PtsAssisted2s
PtsUnassisted2s
PtsAssisted3s
PtsUnassisted3s
PtsPutbacks
HeaveAttempts
NonHeaveArc3FGA
NonHeaveArc3FGM
Fg2aBlocked
TwoPtAssists
ThreePtAssists
Assists
Arc3Assists
Corner3Assists
AtRimAssists
ShortMidRangeAssists
LongMidRangeAs

In [35]:
stats = player_stats_df[["Name", "GamesPlayed", "Usage", "Points", "Turnovers", "BadPassTurnovers", "BadPassOutOfBoundsTurnovers", "FG2A", "FG3A", "ThreePtShootingFoulsDrawn", "2pt And 1 Free Throw Trips", "3pt And 1 Free Throw Trips", "TwoPtShootingFoulsDrawn", "Technical Free Throw Trips", "FtPoints", "FTA"]]
stats = stats.fillna(0)


In [36]:
stats["TrueFTPoss"] = (stats["TwoPtShootingFoulsDrawn"] - stats["2pt And 1 Free Throw Trips"]) + (stats["ThreePtShootingFoulsDrawn"] - stats["3pt And 1 Free Throw Trips"]) + ((stats["FTA"] - stats["Technical Free Throw Trips"] - stats["2pt And 1 Free Throw Trips"] - stats["3pt And 1 Free Throw Trips"]  - ((stats["TwoPtShootingFoulsDrawn"] - stats["2pt And 1 Free Throw Trips"]) * 2) - ((stats["ThreePtShootingFoulsDrawn"] - stats["3pt And 1 Free Throw Trips"]) * 3))*0.5)

In [37]:
stats["FTPercent"] = stats["FtPoints"] / stats["FTA"]

In [38]:
stats["TruePoints"] = stats["Points"] - (stats["Technical Free Throw Trips"] * stats["FTPercent"])

In [39]:
stats["ScoringTurnovers"] = stats["Turnovers"] - (stats["BadPassTurnovers"] + stats["BadPassOutOfBoundsTurnovers"])

In [40]:
stats["TS"] = 0.5 * (stats["TruePoints"] / (((stats["FG2A"] + stats["FG3A"] + stats["TrueFTPoss"]) * 1)))
stats["PPP"] = 0.5 * (stats["TruePoints"] / (((stats["FG2A"] + stats["FG3A"] + stats["TrueFTPoss"]) * 1) + stats["ScoringTurnovers"]))

In [41]:
stats = stats.sort_values(by="Points", ascending=False)
stats

,Name,GamesPlayed,Usage,Points,Turnovers,BadPassTurnovers,BadPassOutOfBoundsTurnovers,FG2A,FG3A,ThreePtShootingFoulsDrawn,...,TwoPtShootingFoulsDrawn,Technical Free Throw Trips,FtPoints,FTA,TrueFTPoss,FTPercent,TruePoints,ScoringTurnovers,TS,PPP
17,Shai Gilgeous-Alexander,76,34.356725,2484,183,66.0,23.0,1221,435.0,12.0,...,269.0,35.0,601.0,669.0,281.0,0.898356,2452.557549,94.0,0.633081,0.603781
2,Anthony Edwards,79,31.355289,2177,249,84.0,40.0,801,811.0,8.0,...,225.0,16.0,415.0,496.0,208.0,0.836694,2163.612903,125.0,0.594399,0.556199
21,Nikola Jokić,70,29.161206,2071,230,96.0,39.0,1033,330.0,6.0,...,168.0,10.0,361.0,451.0,189.5,0.800443,2062.995565,95.0,0.664411,0.626099
43,Giannis Antetokounmpo,67,35.277081,2036,206,50.0,19.0,1256,63.0,1.0,...,352.0,0.0,436.0,707.0,307.0,0.616690,2036.000000,137.0,0.626076,0.577425
13,Jayson Tatum,72,30.682384,1932,209,80.0,24.0,737,728.0,23.0,...,155.0,27.0,358.0,440.0,178.0,0.813636,1910.031818,105.0,0.581263,0.546348
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,Jae Crowder,9,11.764706,23,4,4.0,0.0,7,15.0,2.0,...,0.0,0.0,5.0,6.0,2.0,0.833333,23.000000,0.0,0.479167,0.479167
495,Justin Minaya,19,10.800000,18,5,1.0,0.0,11,10.0,0.0,...,0.0,0.0,0.0,2.0,1.0,0.000000,18.000000,4.0,0.409091,0.346154
488,Joe Ingles,19,11.721612,15,9,4.0,2.0,8,15.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,3.0,NaN,NaN
499,Stanley Umude,22,13.963964,15,3,1.0,1.0,11,15.0,0.0,...,2.0,0.0,2.0,4.0,2.0,0.500000,15.000000,1.0,0.267857,0.258621


In [42]:
stats = stats[["Name", "GamesPlayed", "Usage", "PPP"]]


In [43]:
stats

,Name,GamesPlayed,Usage,PPP
17,Shai Gilgeous-Alexander,76,34.356725,0.603781
2,Anthony Edwards,79,31.355289,0.556199
21,Nikola Jokić,70,29.161206,0.626099
43,Giannis Antetokounmpo,67,35.277081,0.577425
13,Jayson Tatum,72,30.682384,0.546348
...,...,...,...,...
494,Jae Crowder,9,11.764706,0.479167
495,Justin Minaya,19,10.800000,0.346154
488,Joe Ingles,19,11.721612,NaN
499,Stanley Umude,22,13.963964,0.258621


In [44]:
stats_table = stats.to_json(orient="records")


In [47]:
with open("../public/data.json", "w") as file:
    file.write(stats_table)

In [16]:
stats_30 = stats.head(30)
stats_30 = stats_30.sort_values(by="PPP", ascending=False)
stats_30

,Name,GamesPlayed,Usage,PPP
21,Nikola Jokić,70,29.161206,0.626099
17,Shai Gilgeous-Alexander,76,34.356725,0.603781
48,Kevin Durant,62,28.820875,0.599563
76,Damian Lillard,58,27.460815,0.596279
49,Stephen Curry,70,29.193698,0.591723
16,Zach LaVine,74,25.688682,0.591398
26,Karl-Anthony Towns,72,27.216066,0.585278
22,Austin Reaves,73,23.533483,0.580548
43,Giannis Antetokounmpo,67,35.277081,0.577425
38,Jalen Brunson,65,29.561069,0.576696
